# Mvouti : composite, concessions et carte de synthèse

**Cartomize · Version 1.0**

Durée indicative : 50 minutes.

## Objectifs

Importer les fichiers réels, vérifier leur compatibilité spatiale, composer une image et automatiser les cartes des concessions.

## Apport de Cartomize

Le composite, les limites et les tableaux sont intégrés dans une chaîne réutilisable. Les résultats sont enregistrés sans modifier les fichiers du ZIP.

Données : fichiers fournis dans `Tuto.zip`, importés localement. Consulter [les données et leurs limites](../DATA.md).

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [ ]:
out = run_directory("11_mvouti_concessions")
from course import supplied_data
d = supplied_data()

## 1. Vérifier les sources

Le composite est déjà assemblé et contient six bandes nommées. Le nom du fichier indique 2024–2025 ; aucune date d’acquisition précise ni méthode de composition n’est intégrée. Les facteurs GDAL sont 1 et 0, ce qui ne prouve pas une calibration physique. Le suffixe `cloudmask` ne constitue pas un masque QA fourni.

In [ ]:
concessions=cm.read_file(d['concessions'])
admin=cm.read_file(d['administration'])
dimonika=cm.read_file(d['dimonika'])
with rasterio.open(d['composite']) as src:
    display(pd.DataFrame({'bande':src.indexes,'nom':src.descriptions,'facteur':src.scales,'décalage':src.offsets}))
    original_crs=src.crs
    image_bounds=src.bounds
assert len(concessions)==5
assert len(admin)==12
assert len(dimonika)==1

## 2. Harmoniser les emprises

Les fichiers administratifs et de concessions sont projetés en UTM 33S. Le composite et Dimonika sont en coordonnées géographiques. La limite rectangulaire de l’image est utilisée comme emprise de traitement ; elle ne représente pas la limite officielle du district de Mvouti. Les zones administratives sont celles du fichier fourni, sans présumer leur actualité.

In [ ]:
from shapely.geometry import box
footprint=gpd.GeoDataFrame({'name':['Emprise du composite']},geometry=[box(*image_bounds)],crs=original_crs)
concessions=cm.clip(cm.make_valid(concessions),footprint)
concessions=concessions[['nom_con','geometry']].rename(columns={'nom_con':'Concession'})
concessions['Surface_dans_emprise_ha']=cm.area(concessions,metric_crs=CRS)
concessions.to_file(out/'concessions.gpkg',index=False)
administration=cm.clip(admin,footprint)[['NAME_1','geometry']]
dimonika_local=cm.clip(dimonika,footprint)[['NAME','geometry']]
concessions.drop(columns='geometry').to_csv(out/'surfaces_concessions.csv',index=False)
display(concessions.drop(columns='geometry'))

## 3. Construire un aperçu multibande projeté

Pour ce cours, la grille de travail est de 120 m. Le composite original reste intact. La réduction de résolution sert à l’exploration et à la cartographie de contexte ; elle ne doit pas être utilisée pour compter des petites clairières. Aucun indice à calibration additive n’est calculé.

Pour l’affichage et les groupes exploratoires, l’exercice traite les valeurs nulles communes aux bandes et connectées au bord comme un fond périphérique. C’est une hypothèse explicite à confirmer auprès du producteur. Le paramètre `MASQUER_FOND_NUL_PERIPHERIQUE` permet de la désactiver ; les sources restent intactes et les zéros isolés à l’intérieur sont conservés.

In [ ]:
MASQUER_FOND_NUL_PERIPHERIQUE = True
source_preparation=d['composite']
if MASQUER_FOND_NUL_PERIPHERIQUE:
    rapport_fond=cm.mask_background(source_preparation,out/'composite_fond_masque.tif',border_values=(0,))
    source_preparation=Path(rapport_fond['path'])
    display(rapport_fond)
with rasterio.open(source_preparation) as src:
    bands={role:cm.Band(source_preparation,index=i,scale=1,offset=0,unit='source units') for i,role in enumerate(src.descriptions,1)}
scene=cm.Scene('Mvouti_composite_2024_2025',bands,sensor='producteur_non_documente',acquired=None)
prepared=cm.process_imagery([scene],out/'preparation',aoi=footprint,target_crs=CRS,resolution=120,
    band_order=list(bands),mask_clouds=False,mosaic=False,multiband=True,composition=('nir','red','green'))
scientific=Path(prepared.products[0]['multiband'])
rgba,extent,metadata=cm.read_rgb(scientific,bands=('nir','red','green'))
fig,ax=plt.subplots(figsize=(9,7),layout='constrained')
ax.imshow(rgba,extent=extent)
concessions.to_crs(CRS).boundary.plot(ax=ax,color='#f5eec7',linewidth=.8)
ax.set_title('Mvouti | Composite fourni, 2024–2025\nInfrarouge couleur et concessions, aperçu à 120 m')
ax.set_axis_off();gallery(fig,'11_mvouti');plt.show()

## 4. Superposer et exporter

Les contours administratifs et de conservation décrivent des zonages. Leur superposition ne démontre pas de conflit juridique ni d’impact écologique.

In [ ]:
carte=cm.compose_map([
 {'data':scientific,'name':'Composite 2024–2025','rgb':('nir','red','green')},
 {'data':administration,'name':'Limites administratives','color':'none','edgecolor':'#eeeeee','legend':False},
 {'data':dimonika_local,'name':'Dimonika (WDPA, mai 2026)','color':'none','edgecolor':'#61cfab'},
 {'data':concessions,'name':'Concessions','labels':'Concession','color':'none','edgecolor':'#f0d290'},
],title='Mvouti : contexte forestier',subtitle='Composite fourni 2024–2025 | Aperçu à 120 m',crs=CRS,
credits='Fichiers Tuto | Dimonika : Protected Planet, mai 2026 | Données à documenter',aoi=footprint)
carte.export(out/'mvouti.pdf',dpi=300)
carte.export(out/'mvouti.png',dpi=200)
cm.atlas(carte,concessions,out/'atlas_concessions',name_column='Concession',dpi=150)
assert len(list((out/'atlas_concessions').glob('*.pdf')))==len(concessions)

## 5. Explorer les groupes spectraux

K-moyennes explore les différences de signatures sans références terrain. Les groupes sont conservés sous des libellés neutres ; cette sortie n’est pas une classification validée de forêt primaire, secondaire ou culture.

In [ ]:
groups=cm.cluster_raster(scientific,out/'groupes_spectraux',clusters=5,sample_limit=12000,random_state=42)
fig=show_raster(groups,'Groupes spectraux exploratoires | Mvouti',cmap='tab10',vmin=1,vmax=5)
gallery(fig,'11_groupes');plt.show()

## Exercice

Que faut-il obtenir avant de calculer EVI et de nommer les groupes « forêt dense », « culture » ou « sol nu » ?

<details>
<summary>Éléments de correction</summary>

La provenance du composite, les coefficients et unités des bandes, les règles de qualité, une nomenclature et des références indépendantes. Les bandes red edge sont absentes : NDRE ne peut pas être calculé à partir de ce produit.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.